# Task 2 — Optimizing (alternate submission)

Tries the ideas `03_model` did not cover, to see whether anything beats the current model
(**averaged histogram Naive Bayes, 0.7562**). Writes an *alternate* file, `task2_optimizing.csv`;
`task2_noleak.csv` from `03_model` is left untouched.

1. **Smoother feature densities**: `SmoothNB`, a binned kernel density version of Naive Bayes
2. **Imbalance handling**: SMOTE and balanced class weights (instruction hint)
3. **True stacking**: a logistic-regression meta-model on all models' predictions (instruction hint)
4. **Blends** of the strongest additive models
5. Compare all candidates → write the alternate submission

Same leakage rules as `03_model`: no `player_id`; feature statistics fitted in-fold; settings chosen on separate
tuning folds (seed 2026); reported scores on the standard 5 folds (seed 42); blend, stack and class weights scored with nested CV.

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
from sklearn.metrics import f1_score, log_loss, classification_report
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline as imb_pipeline

from data import load_train, load_test, TARGET, CLASS_NAMES, TASK_DIR, SEED
from evaluate import cv_fit_in_fold, get_folds
from features import FeatureBuilder
from model_zoo import MODEL_VIEW, VIEWS, build
from models import SmoothNB
from postprocess import (blend, fit_blend_weights, fit_class_weights, apply_class_weights, nested_score,
                         stack_features, fit_stacker, nested_stack_score)
from submission import save_submission

ART = TASK_DIR / "artifacts"
ORDER = list(CLASS_NAMES.values())
train, test = load_train(), load_test()
y = train[TARGET].to_numpy()
folds = get_folds(y)                                   # scoring folds (same as 03_model)
tune_folds = get_folds(y, n_splits=3, seed=2026)       # tuning folds
tuned = json.loads((ART / "tuned_params.json").read_text())

# OOF predictions of the six 03_model models (recomputed if missing)
oof = {}
for name in MODEL_VIEW:
    p = ART / "oof" / f"noleak_{name}.npy"
    oof[name] = np.load(p) if p.exists() else cv_fit_in_fold(build(name, tuned[name]["params"]), train, y,
                                                              VIEWS[MODEL_VIEW[name]], folds, verbose=False)[0]
REF = f1_score(y, oof["AvgHistNB"].argmax(1), average="macro")
print(f"reference (03_model final, AvgHistNB): {REF:.4f}")

reference (03_model final, AvgHistNB): 0.7562


## 1. Smoother densities: `SmoothNB`
Histogram Naive Bayes has to choose a bin count: too few bins blur the shape, too many make it noisy (64 bins scored worst in `02_features`).
`SmoothNB` uses 200 fine bins and then smooths each class's counts with a Gaussian kernel. The kernel width `sigma` (in bins) and the
smoothing constant `alpha` are chosen on the **tuning folds**.

In [2]:
grid = []
for sigma in [2, 4, 6, 8, 12, 16]:
    for alpha in [0.5, 2, 8]:
        _, s, _ = cv_fit_in_fold(SmoothNB(sigma=sigma, alpha=alpha), train, y, VIEWS["nb"], tune_folds, verbose=False)
        grid.append({"sigma": sigma, "alpha": alpha, "tuning F1": s})
grid = pd.DataFrame(grid)
display(grid.pivot(index="sigma", columns="alpha", values="tuning F1").round(4))
best = grid.loc[grid["tuning F1"].idxmax()]
SMOOTH = dict(sigma=float(best.sigma), alpha=float(best.alpha))
print("chosen:", SMOOTH)
print("SmoothNB on scoring folds ", end="")
oof["SmoothNB"], _, _ = cv_fit_in_fold(SmoothNB(**SMOOTH), train, y, VIEWS["nb"], folds)
np.save(ART / "oof" / "opt_SmoothNB.npy", oof["SmoothNB"])

alpha,0.5,2.0,8.0
sigma,,,
2,0.7552,0.7554,0.7547
4,0.7567,0.7567,0.7563
6,0.7570,0.7567,0.7558
8,0.7565,0.7565,0.7558
12,0.7557,0.7557,0.7552
16,0.7547,0.7548,0.7543


chosen: {'sigma': 6.0, 'alpha': 0.5}
SmoothNB on scoring folds 

macro F1 0.7570  (folds 0.7570 ± 0.0023)  [33 feats, 4s]


## 2. Imbalance handling (applied to tuned LightGBM)
- **Balanced class weights**: rarer classes count more in training.
- **SMOTE**: creates synthetic minority-class rows by interpolating between neighbours. It runs **inside each training fold only**
  (via an imbalanced-learn pipeline), so no synthetic row is built from validation data. SMOTE cannot handle missing values,
  so a median imputer runs first.

In [3]:
lgb_params = tuned["LightGBM"]["params"]
variants = {
    "LightGBM + balanced weights": build("LightGBM", {**lgb_params, "class_weight": "balanced"}),
    "LightGBM + SMOTE": imb_pipeline(SimpleImputer(strategy="median"), SMOTE(random_state=SEED), build("LightGBM", lgb_params)),
}
print(f"{'LightGBM (03_model)':28s} macro F1 {f1_score(y, oof['LightGBM'].argmax(1), average='macro'):.4f}")
for name, model in variants.items():
    print(f"{name:28s}", end=" ")
    oof[name], _, _ = cv_fit_in_fold(model, train, y, VIEWS["gb"], folds)

LightGBM (03_model)          macro F1 0.7537
LightGBM + balanced weights  

macro F1 0.7497  (folds 0.7497 ± 0.0020)  [41 feats, 27s]
LightGBM + SMOTE             

macro F1 0.7521  (folds 0.7521 ± 0.0024)  [41 feats, 39s]


## 3. True stacking
A logistic-regression **meta-model** learns how to combine all models' log-probabilities, instead of using fixed blend weights.
Nested CV: the meta-model is trained on 4 folds of out-of-fold predictions and scored on the 5th.

In [4]:
level1 = ["AvgHistNB", "SmoothNB", "SplineLR", "XGBoost", "LightGBM", "CatBoost", "RandomForest"]
stack_rows = []
for C in [0.01, 0.1, 1.0]:
    for cw in [False, True]:
        stack_rows.append({"C": C, "class weights": cw,
                           "nested F1": nested_stack_score([oof[m] for m in level1], y, folds, use_cw=cw, C=C)})
stack_rows = pd.DataFrame(stack_rows)
display(stack_rows.round(4))

,C,class weights,nested F1
0,0.01,False,0.7560
1,0.01,True,0.7569
2,0.10,False,0.7560
3,0.10,True,0.7567
4,1.00,False,0.7560
5,1.00,True,0.7563


## 4. Blends of the strongest additive models

In [5]:
blend_sets = {
    "SmoothNB + SplineLR": ["SmoothNB", "SplineLR"],
    "AvgHistNB + SplineLR": ["AvgHistNB", "SplineLR"],
    "SmoothNB + AvgHistNB + SplineLR": ["SmoothNB", "AvgHistNB", "SplineLR"],
    "SmoothNB + SplineLR + XGBoost": ["SmoothNB", "SplineLR", "XGBoost"],
}
blend_rows = []
for name, members in blend_sets.items():
    ps = [oof[m] for m in members]
    blend_rows.append({"blend": name, "blend only": nested_score(ps, y, folds, use_cw=False),
                       "blend + class weights": nested_score(ps, y, folds, use_cw=True)})
blend_rows = pd.DataFrame(blend_rows).set_index("blend")
display(blend_rows.round(4))

,blend only,blend + class weights
blend,,
SmoothNB + SplineLR,0.7570,0.7574
AvgHistNB + SplineLR,0.7563,0.7566
SmoothNB + AvgHistNB + SplineLR,0.7570,0.7574
SmoothNB + SplineLR + XGBoost,0.7553,0.7553


## 5. Compare everything
Fold-to-fold variation is about ±0.002. With this many candidates, the top score is also slightly optimistic just by being the
maximum, so a candidate is only adopted if it beats the reference by **more than 0.001**.

In [6]:
cands = []  # (name, kind, members, use_cw, extra, score)
cands.append(("AvgHistNB (reference)", "single", ["AvgHistNB"], False, None, REF))
cands.append(("SmoothNB", "single", ["SmoothNB"], False, None, f1_score(y, oof["SmoothNB"].argmax(1), average="macro")))
for v in variants:
    cands.append((v, "single", [v], False, None, f1_score(y, oof[v].argmax(1), average="macro")))
r = stack_rows.loc[stack_rows["nested F1"].idxmax()]
cands.append((f"Stacking (C={r.C}, class weights={r['class weights']})", "stack", level1, bool(r["class weights"]), float(r.C), float(r["nested F1"])))
for name, members in blend_sets.items():
    for post in ["blend only", "blend + class weights"]:
        cands.append((f"{name} ({post})", "blend", members, post == "blend + class weights", None, float(blend_rows.loc[name, post])))
summary = pd.DataFrame(cands, columns=["candidate", "kind", "members", "class weights", "C", "macro F1"])
summary["vs reference"] = summary["macro F1"] - REF
summary = summary.sort_values("macro F1", ascending=False).reset_index(drop=True)
display(summary[["candidate", "macro F1", "vs reference"]].round(4))

top = summary.iloc[0]
chosen = top if top["vs reference"] > 0.001 else summary[summary.candidate == "AvgHistNB (reference)"].iloc[0]
print(f"\nCHOSEN: {chosen.candidate}   macro F1 {chosen['macro F1']:.4f}   ({chosen['vs reference']:+.4f} vs reference)")

,candidate,macro F1,vs reference
0,SmoothNB + SplineLR (blend + class weights),0.7574,0.0011
1,SmoothNB + AvgHistNB + SplineLR (blend + class...,0.7574,0.0011
2,SmoothNB + AvgHistNB + SplineLR (blend only),0.7570,0.0008
3,SmoothNB + SplineLR (blend only),0.7570,0.0008
4,SmoothNB,0.7570,0.0007
5,"Stacking (C=0.01, class weights=True)",0.7569,0.0007
6,AvgHistNB + SplineLR (blend + class weights),0.7566,0.0004
7,AvgHistNB + SplineLR (blend only),0.7563,0.0001
8,AvgHistNB (reference),0.7562,0.0000
9,SmoothNB + SplineLR + XGBoost (blend only),0.7553,-0.0009



CHOSEN: SmoothNB + SplineLR (blend + class weights)   macro F1 0.7574   (+0.0011 vs reference)


## 6. Fit on all training data → alternate submission

In [7]:
def fit_predict_test(name):
    if name == "SmoothNB":
        model, view = SmoothNB(**SMOOTH), VIEWS["nb"]
    elif name in variants:
        model, view = variants[name], VIEWS["gb"]
    else:
        model, view = build(name, tuned[name]["params"]), VIEWS[MODEL_VIEW[name]]
    fb = FeatureBuilder(**view).fit(train)
    Xtr, Xte = fb.transform(train), fb.transform(test)
    assert "player_id" not in Xtr.columns
    return model.fit(Xtr, y).predict_proba(Xte)

members = list(chosen.members)
test_probs = [fit_predict_test(m) for m in members]
oof_probs = [oof[m] for m in members]
if chosen.kind == "stack":
    meta = fit_stacker(oof_probs, y, C=chosen.C)
    p_oof, p_test = meta.predict_proba(stack_features(oof_probs)), meta.predict_proba(stack_features(test_probs))
else:
    w = fit_blend_weights(oof_probs, y) if len(members) > 1 else np.array([1.0])
    p_oof, p_test = blend(oof_probs, w), blend(test_probs, w)
cw = fit_class_weights(p_oof, y)[0] if chosen["class weights"] else np.ones(4)
test_pred = apply_class_weights(p_test, cw)

noleak = pd.read_csv(TASK_DIR / "submissions" / "task2_noleak.csv")
print(f"test rows that differ from task2_noleak.csv: {np.mean(test_pred != noleak.task2.values):.1%}")
display(pd.Series(test_pred).map(CLASS_NAMES).value_counts(normalize=True).loc[ORDER].round(3).to_frame("test share"))
full = save_submission(test_pred, "optimizing")
print("saved submissions/task2_optimizing.csv and submissions/full_optimizing.csv")

test rows that differ from task2_noleak.csv: 1.9%


,test share
Casual,0.424
Grinder,0.251
Social,0.183
Whale,0.142


saved submissions/task2_optimizing.csv and submissions/full_optimizing.csv


## Summary
See the table in section 5. Every idea is scored with the same leak-free rules as `03_model`, so the numbers compare directly
and can go straight into the report, including the ones that did not help.